# Notebook 3: Using the Claude, OpenAI and Gemini APIs

**Who this is for:** Digital Humanities students with no prior AI or programming background.

**What you will learn**
1. What an API is, and how to keep your API key safe
2. How to send your first prompt to Claude, OpenAI (GPT) and Gemini
3. How to steer a model with a *system prompt*
4. How *temperature* changes the output
5. How *top-p* and *top-k* limit the model's word choices
6. How *few-shot examples* and *structured output* make answers more reliable

You do **not** need all three providers. Pick one, get a key, and run only that section. The helper functions are written so you can compare providers if you wish.

> **Reading the code:** lines that start with `#` are *comments*. Python ignores them; they are notes written for humans to explain what the code does.

## 0. What is an API, and how do I get a key?

An **API** lets your code talk to a model hosted by a company. You send text; you get text back. Providers charge per use (usually a small fraction of a cent for the short prompts in this notebook).

Create a key on the provider's website:

| Provider | Where to create a key | Colab secret name |
|---|---|---|
| Anthropic (Claude) | console.anthropic.com | `ANTHROPIC_API_KEY` |
| OpenAI (GPT) | platform.openai.com | `OPENAI_API_KEY` |
| Google (Gemini) | aistudio.google.com | `GEMINI_API_KEY` |

**Keep keys secret.** Never paste a key directly into a notebook you will share. In Colab, click the **key icon** in the left sidebar, add a secret with the name from the table, and switch on *Notebook access*.

In [ ]:
# Install the three libraries (one per provider). "-q" keeps the output quiet.
!pip install -q anthropic openai google-genai

In [ ]:
import os                      # lets us read environment variables
from getpass import getpass    # asks for text without showing it on screen

def get_key(name):
    """Read a key from Colab Secrets; if missing, ask for it (input is hidden)."""
    try:
        # Colab's built-in secret storage (the key icon in the sidebar)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        # Not in Colab, or secret not set: try an environment variable,
        # otherwise ask the user to type it. "or None" gives None if left empty.
        return os.environ.get(name) or getpass(f"Enter {name} (leave empty to skip): ") or None

# Load one key per provider (any key you do not have will simply be None)
ANTHROPIC_API_KEY = get_key("ANTHROPIC_API_KEY")
OPENAI_API_KEY = get_key("OPENAI_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

# Print True/False only. Never print the keys themselves.
print("Claude key found:", bool(ANTHROPIC_API_KEY))
print("OpenAI key found:", bool(OPENAI_API_KEY))
print("Gemini key found:", bool(GEMINI_API_KEY))

### Choose your models

Providers release new models often and retire old ones. The names below are **settings you can change**. Check the provider's documentation for current model names if you get a "model not found" error.

In [ ]:
# Model names are just text settings. Change them if a name is retired or you want another model.
CLAUDE_MODEL = "claude-sonnet-4-5"      # see docs.claude.com for current model names
OPENAI_MODEL = "gpt-4o-mini"            # see platform.openai.com/docs/models
GEMINI_MODEL = "gemini-2.5-flash"       # see ai.google.dev/gemini-api/docs/models

## 1. Your first prompt

A **prompt** is the text you send to the model. We start with one question to each provider.

In [ ]:
# The text we send to the model
prompt = "In two sentences, explain what the Digital Humanities are."

# --- Claude ---
if ANTHROPIC_API_KEY:                                  # run only if a Claude key exists
    import anthropic
    claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)   # create the connection ("client")
    response = claude.messages.create(
        model=CLAUDE_MODEL,                            # which model answers
        max_tokens=300,                                # maximum length of the answer
        messages=[{"role": "user", "content": prompt}],  # the conversation: one user message
    )
    print("CLAUDE:\n", response.content[0].text)       # the answer text is inside the response

In [ ]:
# --- OpenAI ---
if OPENAI_API_KEY:                                     # run only if an OpenAI key exists
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_API_KEY)     # create the connection ("client")
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL,                            # which model answers
        messages=[{"role": "user", "content": prompt}],  # the conversation: one user message
    )
    print("OPENAI:\n", response.choices[0].message.content)  # the answer text

In [ ]:
# --- Gemini ---
if GEMINI_API_KEY:                                     # run only if a Gemini key exists
    from google import genai
    gemini = genai.Client(api_key=GEMINI_API_KEY)      # create the connection ("client")
    response = gemini.models.generate_content(
        model=GEMINI_MODEL,                            # which model answers
        contents=prompt,                               # the text we send
    )
    print("GEMINI:\n", response.text)                  # the answer text

The three calls look slightly different, but the idea is identical: **send text, receive text**.

### One helper function for all providers

To keep the rest of the notebook short, we wrap the three calls in a single function, `ask()`. You only need to understand what goes in (the prompt and optional settings) and what comes out (text).

Besides the prompt, `ask()` accepts these optional settings (explained later in this notebook):

| Setting | Meaning |
|---|---|
| `system` | standing instructions for the model (section 2) |
| `temperature` | how random the answer is (section 3) |
| `top_p`, `top_k` | limits on the word choices the model may pick from (section 4) |
| `max_tokens` | maximum length of the answer |

In [ ]:
def ask(prompt, provider="claude", system=None, temperature=None,
        top_p=None, top_k=None, max_tokens=500):
    """Send a prompt to the chosen provider and return the answer as text.
    Any setting left as None is not sent, so the provider's default is used."""

    # ---------- Claude ----------
    if provider == "claude":
        # Required parts of the request
        kwargs = dict(model=CLAUDE_MODEL, max_tokens=max_tokens,
                      messages=[{"role": "user", "content": prompt}])
        # Add optional settings only when the user gave them
        if system is not None: kwargs["system"] = system
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None: kwargs["top_p"] = top_p
        if top_k is not None: kwargs["top_k"] = top_k
        # "**kwargs" unpacks the dictionary into named arguments
        return claude.messages.create(**kwargs).content[0].text

    # ---------- OpenAI ----------
    if provider == "openai":
        messages = []
        # OpenAI takes the system prompt as the first message in the list
        if system: messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        kwargs = dict(model=OPENAI_MODEL, messages=messages, max_tokens=max_tokens)
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None: kwargs["top_p"] = top_p
        # OpenAI has no top_k setting, so top_k is ignored for this provider
        return openai_client.chat.completions.create(**kwargs).choices[0].message.content

    # ---------- Gemini ----------
    if provider == "gemini":
        from google.genai import types
        # Gemini groups all settings in one "config" object (None values are skipped)
        config = types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            max_output_tokens=max_tokens,
        )
        return gemini.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config).text

    # If we reach this line, the provider name was not recognised
    raise ValueError("provider must be 'claude', 'openai' or 'gemini'")

# Pick the provider you have a key for:
PROVIDER = "claude"   # or "openai" or "gemini"
print(ask("List three tasks in Digital Humanities where a language model can help a researcher.", provider=PROVIDER))

## 2. System prompts: setting the role and style

A **system prompt** gives the model standing instructions: who it is, who it is talking to, and how to answer. The same question can produce very different answers.

In [ ]:
question = "What is OCR, and why does it make mistakes on old printed texts?"      # the same question every time

# 1) No instructions: the model chooses its own style
print("--- Default ---")
print(ask(question, provider=PROVIDER))

# 2) System prompt that sets audience, format and length
print("\n--- As a tutor for first-year Digital Humanities students, in 3 short bullet points ---")
print(ask(
    question,
    provider=PROVIDER,
    system="You are a friendly tutor for first-year Digital Humanities students. "
           "Answer in exactly 3 short bullet points and avoid jargon.",
))

# 3) A different system prompt gives a different role and tone
print("\n--- As a careful digital archivist ---")
print(ask(
    question,
    provider=PROVIDER,
    system="You are a careful digital archivist. Mention practical limits and common pitfalls "
           "for people digitising historical collections. Keep the answer under 80 words.",
))

**Lesson:** Be explicit about the audience, the format and the length you want.

## 3. Temperature: how "creative" should the model be?

A language model writes **one small piece of text (a *token*, roughly a word or part of a word) at a time**. At every step it gives each possible next token a probability, then picks one.

*Temperature* controls how adventurous that pick is. Low values give consistent, predictable answers (good for data extraction). High values give more varied answers (good for brainstorming). Run the cell several times to see the difference.

> Some newer models fix the temperature or ignore this setting. If you see an error, set `temperature=None`.

In [ ]:
# A prompt where many different answers are possible
creative_prompt = "Suggest a title for an article about digitising Arabic manuscripts. Give only the title."

for temp in [0.0, 1.0]:                          # try a low and a high temperature
    print(f"Temperature = {temp}")
    for _ in range(3):                           # ask three times to see how much answers vary
        # "_" is a throwaway loop variable: we only care about repeating 3 times
        print("  -", ask(creative_prompt, provider=PROVIDER, temperature=temp, max_tokens=60))
    print()

At temperature 0 the three answers are (almost) identical. At 1.0 they differ. For research tasks such as classification or extraction, **prefer low temperature** so your results are reproducible.

## 4. Top-p and top-k: limiting the word choices

Temperature changes *how boldly* the model picks. **Top-k** and **top-p** instead *shorten the list of candidates* before the pick happens.

Imagine the model has finished the sentence *"The manuscript was written in ..."* and scores the next word like this:

| Candidate | Probability |
|---|---|
| Arabic | 45% |
| Latin | 25% |
| Greek | 15% |
| Syriac | 8% |
| all other words together | 7% |

**Top-k: keep only the k most likely candidates.**
- `top_k = 1`: only *Arabic* is allowed. The answer is always the single most likely word.
- `top_k = 3`: *Arabic*, *Latin* and *Greek* are allowed. The model picks among these three.

**Top-p (also called "nucleus sampling"): keep the smallest group of candidates whose probabilities add up to at least p.**
- `top_p = 0.4`: *Arabic* alone already reaches 45%, so only *Arabic* stays.
- `top_p = 0.7`: *Arabic* + *Latin* = 70%, so these two stay.
- `top_p = 0.95`: *Arabic*, *Latin*, *Greek* and *Syriac* are all needed to pass 95%, so all four stay.

The difference: top-k always keeps a **fixed number** of candidates; top-p keeps a **variable number**, fewer when the model is confident and more when it is unsure.

| | Lower value | Higher value |
|---|---|---|
| `top_k` | safer, more repetitive | more variety |
| `top_p` (0 to 1) | safer, more repetitive | more variety |

**Which providers support them?**

| Provider | temperature | top_p | top_k |
|---|---|---|---|
| Claude | yes | yes | yes |
| OpenAI | yes | yes | **no** |
| Gemini | yes | yes | yes |

**Practical advice**
- Change **one** setting at a time. Temperature, top-p and top-k all affect randomness, so combining them makes results hard to understand.
- Some newer models do not accept temperature and top-p together, or ignore these settings. If you get an error, set one of them to `None`.
- For research that must be reproducible, use a low temperature and note the settings you used.

In [ ]:
# Same prompt as before, so we can see only the effect of top_p and top_k.
# We leave temperature unset (None) so that only ONE setting changes at a time.
creative_prompt = "Suggest a title for an article about digitising Arabic manuscripts. Give only the title."

# ---- top_p: low value = very few candidates, high value = many candidates ----
for p in [0.1, 0.95]:
    print(f"top_p = {p}")
    for _ in range(3):                                   # three tries to see the variety
        print("  -", ask(creative_prompt, provider=PROVIDER, top_p=p, max_tokens=60))
    print()

# ---- top_k: only available for Claude and Gemini ----
if PROVIDER == "openai":
    print("OpenAI does not offer top_k. Switch PROVIDER to 'claude' or 'gemini' to try it.")
else:
    for k in [1, 40]:                                    # 1 = always the most likely word
        print(f"top_k = {k}")
        for _ in range(3):
            print("  -", ask(creative_prompt, provider=PROVIDER, top_k=k, max_tokens=60))
        print()

**What to look for:** with a low `top_p` or with `top_k = 1`, the three titles should be (almost) the same. With a high `top_p` or a larger `top_k`, they should vary more. Providers may handle these settings slightly differently, so the effect can be stronger or weaker depending on the model.

## 5. Few-shot prompting: teach by example

Instead of only describing a task, you can **show examples** of the input and the answer you expect. This is called *few-shot prompting*, and it is one of the most useful techniques for researchers.

Task: label the **document type** of short snippets from a digitised archive. This is the kind of metadata a cataloguer adds to each item, and a model can give a useful first draft for thousands of items.

In [ ]:
# The prompt contains the instructions, three solved examples, and finally the
# new text to classify. The model continues the pattern after the last "Label:".
few_shot_prompt = """Classify each archive snippet as one of: LETTER, NEWSPAPER, POEM, LEGAL_DEED.
Answer with the label only.

Text: "Dear Mother, I hope this letter finds you well. The voyage was long."
Label: LETTER

Text: "The city council announced on Tuesday that the old market will be restored."
Label: NEWSPAPER

Text: "I, the undersigned, hereby sell to Ahmad ibn Ali the house in the old quarter for two hundred dinars."
Label: LEGAL_DEED

Text: "The moon walks softly over the sleeping sea."
Label:"""

# temperature=0 for a stable answer; max_tokens=10 because we only need one label
print(ask(few_shot_prompt, provider=PROVIDER, temperature=0, max_tokens=10))

It also works for Arabic input. The instructions and labels stay in English, but the text to classify is Arabic.

In [ ]:
# Keep everything before the last "Text:" (instructions + examples), then add an Arabic text.
# rsplit('Text:', 1) splits the string at the LAST occurrence of "Text:".
arabic_prompt = few_shot_prompt.rsplit('Text:', 1)[0] + """Text: "اشترى أحمد بن علي الدار الكائنة في الحي القديم بمائتي دينار."
Label:"""

print(ask(arabic_prompt, provider=PROVIDER, temperature=0, max_tokens=10))

## 6. Structured output: getting answers you can analyse

For research you often want results in a **table**, not a paragraph. A typical Digital Humanities example is turning free-text catalogue descriptions into **structured metadata** (author, date, place, language...). Ask the model to answer in **JSON** (a simple, standard format) and read it with Python.

In [ ]:
import json    # Python's built-in tool for reading JSON text

# A free-text description, as found in many archive catalogues (this example is invented)
text = ("Letter from Mary Hall to her brother Thomas, written in Alexandria on 14 March 1887. "
        "Four pages, in English, held in a private family collection.")

# We describe the exact format we want. "\"" writes a quotation mark inside a text string.
structured_prompt = (
    "Extract catalogue metadata from the description.\n"
    "Return ONLY valid JSON: one object with the keys \"author\", \"recipient\", \"place\", "
    "\"date\", \"pages\" and \"language\". Use null when a value is not stated.\n\n"
    "Text: " + text
)

raw = ask(structured_prompt, provider=PROVIDER, temperature=0, max_tokens=400)
print("Raw answer from the model:\n", raw)

In [ ]:
import pandas as pd    # pandas turns lists of records into tables

def parse_json(raw):
    """Models sometimes wrap JSON in ```json fences. Remove them before parsing."""
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")                     # remove the backtick fences
        # drop the leading word "json" and the line break after it, if present
        cleaned = cleaned[cleaned.find("\n") + 1:] if cleaned.lower().startswith("json") else cleaned
    return json.loads(cleaned.strip())                   # turn the JSON text into Python data

data = parse_json(raw)           # a Python dictionary, e.g. {"author": ..., "place": ...}
pd.DataFrame([data])             # one record becomes one table row; many records would give many rows

Now the answer is a table you can sort, filter, count or export to a spreadsheet. With thousands of catalogue entries, this turns hours of manual data entry into a first draft that you then check. Notebook 8 uses the same pattern for entity and relationship extraction.

## Good habits when using LLMs in research

- **Verify.** Models can state wrong facts confidently ("hallucination"). Check important claims against reliable sources.
- **Record everything.** Keep the exact prompt, the model name, the settings (temperature, top-p, top-k) and the date. Models change over time.
- **Protect data.** Do not send private, sensitive or unpublished material to a cloud API unless your ethics approval allows it.
- **Mind the cost.** Test on 5 items before running on 5,000.

## Exercises

1. Change the system prompt in section 2 to answer as a museum guide for children. How does the answer change?
2. In section 4, try `top_p = 0.5` and `top_p = 1.0`. At which value do the titles start to differ?
3. Add a fifth document type (for example `MAP_DESCRIPTION`) to the few-shot example, with an example for it.
4. In section 6, paste a catalogue description from your own research area and add a field such as `subject`.
5. Run the same prompt on two providers. Where do the answers differ?

**Next:** Notebook 8 applies these ideas to translation, summarisation, sentiment analysis, named entity recognition and relationship extraction.